# Evaluación del modelo y análisis del umbral de clasificación

## Objetivos de la sesión

En esta sesión se busca:

1. Retomar el modelo baseline de regresión logística.
2. Profundizar en la evaluación del clasificador.
3. Comparar distintas métricas de desempeño.
4. Analizar el efecto del umbral de clasificación sobre los resultados.
5. Comprender la diferencia entre predecir probabilidades y predecir clases.
6. Identificar fortalezas y limitaciones del modelo baseline.

## Contexto

En la sesión anterior se entrenó una regresión logística para predecir incumplimiento de pago. Sin embargo, evaluar un clasificador binario no consiste únicamente en calcular el accuracy.

En problemas de riesgo de crédito, distintas métricas pueden reflejar comportamientos muy diferentes del modelo, especialmente cuando las clases no están perfectamente balanceadas. Además, la decisión final depende del umbral que se utilice para convertir probabilidades en etiquetas de clase.

Importamos las librerias necesarias.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Carga del dataset limpio

En esta sección se carga la versión depurada del conjunto de datos preparada en sesiones anteriores.

Además, se verifica que la variable objetivo tenga el nombre `default_next_month` y, si es necesario, se reconstruye la variable `AGE_GROUP` para usarla como predictor categórico en el modelo.

In [2]:
ruta_csv = "../data/default_credit_clean.csv"

df = pd.read_csv(ruta_csv)

if "default payment next month" in df.columns:
    df = df.rename(columns={"default payment next month": "default_next_month"})

if "AGE_GROUP" not in df.columns:
    bins = [20, 30, 40, 50, 60, 80]
    labels = ["20-29", "30-39", "40-49", "50-59", "60+"]
    df["AGE_GROUP"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras columnas:")
print(df.columns.tolist()[:10])
df.head()

Dimensiones del dataset: (30000, 29)

Primeras columnas:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_next_month,SEX_LABEL,EDUCATION_LABEL,MARRIAGE_LABEL,AGE_GROUP
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1,Femenino,Universidad,Casado,20-29
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1,Femenino,Universidad,Soltero,20-29
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0,Femenino,Universidad,Soltero,30-39
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0,Femenino,Universidad,Casado,30-39
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0,Masculino,Universidad,Casado,50-59




El conjunto de datos se cargó correctamente y contiene **30,000 observaciones** y **29 variables**. Esto confirma que estamos trabajando con la base completa y que ya incluye algunas columnas auxiliares útiles, como:

- `SEX_LABEL`
- `EDUCATION_LABEL`
- `MARRIAGE_LABEL`
- `AGE_GROUP`

Además, se observa que la variable objetivo ya aparece con el nombre **`default_next_month`**, lo cual facilita su uso en el modelado.

Las primeras filas muestran que el dataset contiene:

- variables demográficas (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`),
- variables de historial de pago (`PAY_0`, `PAY_2`, ..., `PAY_6`),
- variables de facturación (`BILL_AMT1`, ..., `BILL_AMT6`),
- variables de pagos previos (`PAY_AMT1`, ..., `PAY_AMT6`),
- y la variable objetivo de incumplimiento.

Con esto, el conjunto de datos está listo para separar predictores y variable objetivo.

## Separación de predictores y variable objetivo

En esta sección se define:

- `X`: conjunto de variables predictoras,
- `y`: variable objetivo `default_next_month`.

Además, se excluyen columnas que no deben entrar directamente al modelo, como:

- el identificador `ID`,
- y las etiquetas de texto auxiliares (`SEX_LABEL`, `EDUCATION_LABEL`, `MARRIAGE_LABEL`).

Estas columnas son útiles para interpretación, pero no para el ajuste del modelo.

In [3]:
cols_to_drop = ["default_next_month"]
optional_drop = ["ID", "SEX_LABEL", "EDUCATION_LABEL", "MARRIAGE_LABEL"]

for col in optional_drop:
    if col in df.columns:
        cols_to_drop.append(col)

X = df.drop(columns=cols_to_drop)
y = df["default_next_month"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("\nColumnas de X:")
print(X.columns.tolist())

Dimensiones de X: (30000, 24)
Dimensiones de y: (30000,)

Columnas de X:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'AGE_GROUP']


## Distribución de la variable objetivo

Antes de entrenar o evaluar el modelo, conviene revisar cómo se distribuyen las clases de la variable objetivo.

Esto permite identificar si el problema está balanceado o desbalanceado, lo cual es importante porque puede afectar la interpretación de métricas como `accuracy`, `precision` y `recall`.

In [4]:
target_dist = y.value_counts().sort_index().to_frame("count")
target_dist["proportion"] = y.value_counts(normalize=True).sort_index()
target_dist

,count,proportion
default_next_month,,
0,23364,0.7788
1,6636,0.2212



La variable objetivo presenta dos clases:

- **0 = no incumplimiento:** 23,364 casos (**77.88%**)
- **1 = incumplimiento:** 6,636 casos (**22.12%**)

Esto muestra que el problema está **desbalanceado**, ya que la mayoría de las observaciones corresponde a clientes que **no incumplieron**. Aunque la clase positiva no es extremadamente rara, sí es claramente minoritaria.

Este punto es importante porque, en conjuntos desbalanceados, una métrica como el **accuracy** puede dar una impresión demasiado optimista del modelo. Por ello, en esta sesión será necesario revisar también otras métricas, como:

- **precision**,
- **recall**,
- **F1-score**,
- y **ROC-AUC**.

En particular, como el interés del problema está en detectar clientes con riesgo de incumplimiento, será fundamental observar qué tan bien identifica el modelo la clase `1`.

## Visualización de la variable objetivo

Además de la tabla de frecuencias, es útil visualizar la distribución de la variable objetivo mediante una gráfica de barras.

Esto permite apreciar de forma más inmediata el desbalance entre las clases.

In [ ]:
y.value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No default", "Default"], rotation=0)
plt.ylabel("Frecuencia")
plt.title("Distribución de la variable objetivo")
plt.show()